# Deep Neural Network for MNIST Classification

We'll apply all the knowledge from the lectures in this section to write a deep neural network. The problem we've chosen is referred to as the "Hello World" of deep learning because for most students it is the first deep learning algorithm they see.

The dataset is called MNIST and refers to handwritten digit recognition. You can find more about it on Yann LeCun's website (Director of AI Research, Facebook). He is one of the pioneers of what we've been talking about and of more complex approaches that are widely used today, such as covolutional neural networks (CNNs). 

The dataset provides 70,000 images (28x28 pixels) of handwritten digits (1 digit per image). 

The goal is to write an algorithm that detects which digit is written. Since there are only 10 digits (0, 1, 2, 3, 4, 5, 6, 7, 8, 9), this is a classification problem with 10 classes. 

Our goal would be to build a neural network with 2 hidden layers.

## Import the relevant packages

In [2]:
import numpy as np
import tensorflow as tf

# TensorFLow includes a data provider for MNIST that we'll use.
# It comes with the tensorflow-datasets module, therefore, if you haven't please install the package using
# pip install tensorflow-datasets 
# or
# conda install tensorflow-datasets

import tensorflow_datasets as tfds

# these datasets will be stored in C:\Users\*USERNAME*\tensorflow_datasets\...
# the first time you download a dataset, it is stored in the respective folder 
# every other time, it is automatically loading the copy on your computer 

## Data

That's where we load and preprocess our data.

In [3]:
# remember the comment from above
# these datasets will be stored in C:\Users\*USERNAME*\tensorflow_datasets\...
# the first time you download a dataset, it is stored in the respective folder 
# every other time, it is automatically loading the copy on your computer 

# tfds.load actually loads a dataset (or downloads and then loads if that's the first time you use it) 
# in our case, we are interesteed in the MNIST; the name of the dataset is the only mandatory argument
# there are other arguments we can specify, which we can find useful
# mnist_dataset = tfds.load(name='mnist', as_supervised=True)
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)
# with_info=True will also provide us with a tuple containing information about the version, features, number of samples
# we will use this information a bit below and we will store it in mnist_info

# as_supervised=True will load the dataset in a 2-tuple structure (input, target) 
# alternatively, as_supervised=False, would return a dictionary
# obviously we prefer to have our inputs and targets separated 

In [4]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples,  tf.int64)

def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255.
    return image, label

scaled_train_and_validation_data = mnist_train.map(scale)
scaled_test_data = mnist_test.map(scale)

BUFFER_SIZE = 10000

shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

BATCH_SIZE = 100

train_data = train_data.batch(BATCH_SIZE)
validation_data = validation_data.batch(num_validation_samples)
test_data = scaled_test_data.batch(num_test_samples)

validation_inputs, validation_targets = next(iter(validation_data))

In [5]:
input_size = 784
output_size = 10
hidden_layer_size = 150

model = tf.keras.Sequential([
                             tf.keras.layers.Flatten(input_shape=(28,28,1)),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(output_size, activation='softmax')
                             ])

C:\Users\mvgarcia\.conda\envs\py312-TF2.0\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
##Choose the optimizer and the loss function

In [7]:
custom_optimizer = tf.keras.optimizers.Adam() #default learning rate
#model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.compile(optimizer=custom_optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

##Training

In [8]:
NUM_EPOCHS = 10
model.fit(train_data, epochs = NUM_EPOCHS, validation_data=(validation_inputs, validation_targets), verbose=2)

Epoch 1/10
540/540 - 6s - 12ms/step - accuracy: 0.9128 - loss: 0.2917 - val_accuracy: 0.9537 - val_loss: 0.1614
Epoch 2/10
540/540 - 3s - 6ms/step - accuracy: 0.9644 - loss: 0.1164 - val_accuracy: 0.9692 - val_loss: 0.0997
Epoch 3/10
540/540 - 4s - 7ms/step - accuracy: 0.9751 - loss: 0.0798 - val_accuracy: 0.9743 - val_loss: 0.0806
Epoch 4/10
540/540 - 4s - 7ms/step - accuracy: 0.9810 - loss: 0.0607 - val_accuracy: 0.9798 - val_loss: 0.0628
Epoch 5/10
540/540 - 4s - 7ms/step - accuracy: 0.9843 - loss: 0.0502 - val_accuracy: 0.9843 - val_loss: 0.0528
Epoch 6/10
540/540 - 4s - 7ms/step - accuracy: 0.9867 - loss: 0.0396 - val_accuracy: 0.9850 - val_loss: 0.0455
Epoch 7/10
540/540 - 4s - 7ms/step - accuracy: 0.9888 - loss: 0.0342 - val_accuracy: 0.9865 - val_loss: 0.0415
Epoch 8/10
540/540 - 3s - 6ms/step - accuracy: 0.9896 - loss: 0.0316 - val_accuracy: 0.9848 - val_loss: 0.0517
Epoch 9/10
540/540 - 4s - 7ms/step - accuracy: 0.9901 - loss: 0.0300 - val_accuracy: 0.9908 - val_loss: 0.0284


## Result

Apparently, with the default learning rate, 4 hidden layers of 150 units each and increasing to 10 epochs is enough to pass 98.5% 

## Testing
Let's run the model with data it has never seen

In [9]:
test_loss, test_accuracy = model.evaluate(test_data)
print('Test loss: {0:.2f}. Test Accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.9776 - loss: 0.0907
Test loss: 0.09. Test Accuracy: 97.76%


With a 97.76% accuracy with the test data, it is confirmed that the model is well trained and still not overfitted.